In [1]:
tabla='qtsop10'
col_tabla = "solopefec"
dat= "dat_ceqx001_essi"
col_essi='fec_sol'
col_dat='fec_sol'
essi='essi_dat_cqx001_etl'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2
import numpy as np

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge
control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [5]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [6]:
fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

c1= text(query)
connection2.execute(c1)

In [8]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_cas is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

cqxestfis= pd.read_sql_query(f"SELECT id_estfis,cod_efi,des_efi FROM dim_cqxestfis", con=connection2)
cqxestfis['cod_efi']=cqxestfis['cod_efi'].str.strip()

cqxestsolope= pd.read_sql_query(f"SELECT id_estsolope,cod_eso,des_eso FROM dim_cqxestsolope", con=connection2)
cqxestsolope['cod_eso']=cqxestsolope['cod_eso'].str.strip()

cqxestreg= pd.read_sql_query(f"SELECT id_estreg,cod_est,des_est FROM dim_cqxestreg", con=connection2)
cqxestreg['cod_est']=cqxestreg['cod_est'].str.strip()

cqxcenqx= pd.read_sql_query(f"SELECT id_cenqx,ori_cas,cod_cas,cod_cqx FROM dim_cqxcenqx", con=connection2)
cqxcenqx['cod_cqx']=cqxcenqx['cod_cqx'].str.strip()
cqxcenqx['cod_cas']=cqxcenqx['cod_cas'].str.strip()

cqxsalas= pd.read_sql_query(f"SELECT id_sala,cod_ori,cod_cas,cod_cqx,cod_sal FROM dim_cqxsalas", con=connection2)
cqxsalas['cod_sal']=cqxsalas['cod_sal'].str.strip()

areas = pd.read_sql_query(f"SELECT id_area,cod_are FROM dim_areas", con=connection2)
areas['cod_are']=areas['cod_are'].str.strip()

servicios = pd.read_sql_query(f"SELECT id_serv,cod_ser,des_ser FROM dim_servicios", con=connection2)
servicios['cod_ser']=servicios['cod_ser'].str.strip()

emecod= pd.read_sql_query(f"SELECT id_emecod,cod_eme,des_eme FROM dim_emecod", con=connection2)
emecod['cod_eme']=emecod['cod_eme'].str.strip()

cqxtipope= pd.read_sql_query(f"SELECT id_tipope,cod_ope,des_ope FROM dim_cqxtipope", con=connection2)
cqxtipope['cod_ope']=cqxtipope['cod_ope'].str.strip()

cqxrieqx= pd.read_sql_query(f"SELECT id_rieqx,cod_rqx,des_rqx FROM dim_cqxrieqx", con=connection2)
cqxrieqx['cod_rqx']=cqxrieqx['cod_rqx'].str.strip()

cqxmotsus= pd.read_sql_query(f"SELECT id_motsus,cod_msu,des_msu FROM dim_cqxmotsus", con=connection2)
cqxmotsus['cod_msu']=cqxmotsus['cod_msu'].str.strip()

cqxaneste= pd.read_sql_query(f"SELECT id_anestesia,cod_ane,des_ane FROM dim_cqxaneste", con=connection2)
cqxaneste['cod_ane']=cqxaneste['cod_ane'].str.strip()

cqxmopreslab= pd.read_sql_query(f"SELECT id_reslab,cod_res,des_lab FROM dim_cqxmopreslab", con=connection2)
cqxmopreslab['cod_res']=cqxmopreslab['cod_res'].str.strip()

cqxmoprieqx= pd.read_sql_query(f"SELECT id_rieqx,cod_rie,des_rie FROM dim_cqxmoprieqx", con=connection2)
cqxmoprieqx['cod_rie']=cqxmoprieqx['cod_rie'].str.strip()

cqxmoprieneu= pd.read_sql_query(f"SELECT id_rieneu,cod_rieneu,des_rieneu FROM dim_cqxmoprieneu", con=connection2)
cqxmoprieneu['cod_rieneu']=cqxmoprieneu['cod_rieneu'].str.strip()

cqxmopconinf= pd.read_sql_query(f"SELECT id_coninf,cod_coninf,des_coninf FROM dim_cqxmopconinf", con=connection2)
cqxmopconinf['cod_coninf']=cqxmopconinf['cod_coninf'].str.strip()

cqxmopordtra= pd.read_sql_query(f"SELECT id_ordtra,cod_ordtra,des_ordtra FROM dim_cqxmopordtra", con=connection2)
cqxmopordtra['cod_ordtra']=cqxmopordtra['cod_ordtra'].str.strip()

cqxresevapreqx= pd.read_sql_query(f"SELECT id_reseva,cod_reseva,des_reseva FROM dim_cqxresevapreqx", con=connection2)
cqxresevapreqx['cod_reseva']=cqxresevapreqx['cod_reseva'].str.strip()

cqxpriope= pd.read_sql_query(f"SELECT id_priope,cod_pop FROM dim_cqxpriope", con=connection2)
cqxpriope['cod_pop']=cqxpriope['cod_pop'].str.strip()

oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas["ori_cod"]=oricas["ori_cod"].str.strip()

tipdoc = pd.read_sql_query(f"SELECT id_tipdoc,cod_tdo FROM dim_tipdoc", con=connection2)
tipdoc["cod_tdo"]=tipdoc["cod_tdo"].str.strip()

numdoc = pd.read_sql_query(f"SELECT id_person,num_doc FROM dim_personal", con=connection2)
numdoc["num_doc"]=numdoc["num_doc"].str.strip()
numdoc["num_doc"]=numdoc["num_doc"].str.replace(' ', '', regex=True)
numdoc=numdoc.drop_duplicates(subset="num_doc")

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_pro","dt_fecha":"fec_pro","dt_mes":"mes_pro","dt_dia":"dia_pro","dt_dia_sem":"dia_sem_pro","dt_sem":"sem_pro","dt_ano":"ano_pro"})

In [9]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

#INICIO DEL DAT

In [10]:
base1.shape

(68485, 79)

In [11]:
inicioProceso=time.time()

In [12]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

In [13]:
base1.shape

(68485, 79)

# EXCELENTE PRACTICA, YA QUE ASI EL DAT SERÁ MÁS RAPIDO PORQUE PROCESARÁ MENOS COLUMNAS

In [14]:
#Eliminamos las columnas que no se usarán aquí: las descripciones y otras 4 más, previa evaluación

# Lista de columnas a eliminar
columnas_eliminar = ['des_cas', 'des_red', 'des_fis', 'des_sop', 'des_reg', 'des_cqx', 'des_sal',
                     'des_are', 'des_ser', 'des_eme_sol', 'des_are_sol', 'des_ser_sol',
                     'des_tip_ope', 'des_rqx', 'des_mot_sus', 'des_tip_ane', 'des_res_lab',
                     'des_res_rieqx', 'des_res_rieneu', 'des_con_inf', 'des_ord_tra', 'des_res_eva',
                     'sdp_hos_pre', 'sdp_hos_pos', 'req_pro_flg']

# Eliminar las columnas
base1 = base1.drop(columns=columnas_eliminar)


In [15]:
base1.shape

(68485, 54)

In [16]:
control_a.append(base1.shape[0])

In [17]:
cqxcenqx["KEY"]=cqxcenqx["ori_cas"].str.strip()+cqxcenqx["cod_cas"].str.strip()+cqxcenqx["cod_cqx"].str.strip()
cqxcenqx=cqxcenqx.drop(["ori_cas",'cod_cas','cod_cqx'], axis=1)
cqxcenqx = cqxcenqx.rename(columns={"id_cenqx": "id_cqx"})
base1["KEY"]=base1["ori_cas"].str.strip()+base1["cod_cas"].astype(str).str.strip()+base1['cod_cqx'].astype(str).str.strip()
merge=pd.merge(base1,cqxcenqx,how='left',on='KEY')
base1=pd.merge(base1,cqxcenqx,how='inner',on='KEY')
base1.shape

(68484, 56)

In [18]:
log_falla('id_cqx', 'KEY', True)
log_control('dim_cqxcenqx')
base1 = base1.drop("KEY",axis=1)

In [19]:
cqxsalas["KEY"]=cqxsalas["cod_ori"].str.strip()+cqxsalas["cod_cas"].str.strip()+cqxsalas["cod_cqx"].str.strip()+cqxsalas["cod_sal"].str.strip()
cqxsalas=cqxsalas.drop(["cod_ori",'cod_cas','cod_cqx','cod_sal'], axis=1)
base1["KEY"]=base1["ori_cas"].str.strip()+base1["cod_cas"].astype(str).str.strip()+base1['cod_cqx'].astype(str).str.strip()+base1["cod_sal"].str.strip()
merge=pd.merge(base1,cqxsalas,how='left',on='KEY')
base1=pd.merge(base1,cqxsalas,how='left',on='KEY') #left para no perder datos
base1.shape

(68484, 57)

In [20]:
log_falla('id_sala', 'KEY', True)
log_control('dim_cqxsalas')
base1 = base1.drop(["cod_sal","cod_cqx","KEY"],axis=1)

In [21]:
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
base1['ori_cas']=base1['ori_cas'].str.strip()
base1["ori_cas"]=base1["ori_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1.shape

(68484, 55)

In [22]:
log_falla('id_oricas', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [23]:
oricas=oricas.rename(columns={"id_oricas":"id_oricaseva","ori_cas":"ori_cas_eva"})
base1['ori_cas_eva']=base1['ori_cas_eva'].str.strip()
base1["ori_cas_eva"]=base1["ori_cas_eva"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,oricas,how='inner',on='ori_cas_eva')
base1=pd.merge(base1,oricas,how='left',on='ori_cas_eva') #Tiene que ser left o se pierden datos
base1.shape

(68484, 55)

In [24]:
merge.shape

(8095, 55)

In [25]:
log_falla('id_oricaseva', 'ori_cas_eva', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas_eva',axis=1)

In [26]:
base1['cod_cas']=base1['cod_cas'].str.strip()
base1["cod_cas"]=base1["cod_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cas,how='left',on='cod_cas')
base1=pd.merge(base1,cas,how='inner',on='cod_cas')
base1.shape

(68484, 55)

In [27]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [28]:
cas=cas.rename(columns={"id_cas":"id_caseva","cod_cas":"cod_cas_eva"})
base1['cod_cas_eva']=base1['cod_cas_eva'].str.strip()
base1["cod_cas_eva"]=base1["cod_cas_eva"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cas,how='inner',on='cod_cas_eva')
base1=pd.merge(base1,cas,how='left',on='cod_cas_eva') #left para no perder datos
base1.shape

(68484, 55)

In [29]:
log_falla('id_caseva', 'cod_cas_eva', True)
log_control('dim_cas')
base1=base1.drop('cod_cas_eva',axis=1)

In [30]:
base1['cod_red']=base1['cod_red'].str.strip()
base1["cod_red"]=base1["cod_red"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,red,how='left',on='cod_red')
base1=pd.merge(base1,red,how='inner',on='cod_red')
base1.shape

(68484, 55)

In [31]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [32]:
base1['cod_tdi']=base1['cod_tdi'].str.strip()
base1["cod_tdi"]=base1["cod_tdi"].str.replace(' ', '', regex=True)
tipdoc=tipdoc.rename(columns={"id_tipdoc": "id_tdi","cod_tdo":"cod_tdi"})
merge=pd.merge(base1,tipdoc,how='left',on='cod_tdi')
base1=pd.merge(base1,tipdoc,how='inner',on='cod_tdi')
base1.shape

(68484, 55)

In [33]:
log_falla('id_tdi', 'cod_tdi', True)
log_control('dim_tipdoc')
base1=base1.drop('cod_tdi',axis=1)

In [34]:
base1['eva_tip_doc']=base1['eva_tip_doc'].str.strip()
base1["eva_tip_doc"]=base1["eva_tip_doc"].str.replace(' ', '', regex=True)
tipdoc=tipdoc.rename(columns={"id_tdi": "id_evatipdoc","cod_tdi":"eva_tip_doc"})
merge=pd.merge(base1,tipdoc,how='left',on='eva_tip_doc')
base1=pd.merge(base1,tipdoc,how='left',on='eva_tip_doc') #left para no perder datos
base1.shape

(68484, 55)

In [35]:
base1.shape

(68484, 55)

In [36]:
log_falla('id_evatipdoc', 'eva_tip_doc', True)
log_control('dim_tipdoc')
base1=base1.drop('eva_tip_doc',axis=1)

In [37]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68484 entries, 0 to 68483
Data columns (total 54 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   num_sol       68484 non-null  float64            
 1   fec_sol       68484 non-null  object             
 2   act_med       68484 non-null  float64            
 3   num_doc       68484 non-null  object             
 4   fec_sop       68484 non-null  object             
 5   fec_pro       42188 non-null  object             
 6   hor_pro       37609 non-null  object             
 7   est_fis       45871 non-null  object             
 8   est_sop       68484 non-null  object             
 9   est_reg       68484 non-null  object             
 10  usu_cre       68484 non-null  object             
 11  fec_cre       68484 non-null  datetime64[ns, UTC]
 12  usu_mod       42188 non-null  object             
 13  fec_mod       42188 non-null  datetime64[ns, UTC]
 14  est_oc

In [38]:
numdoc=numdoc.rename(columns={"num_doc": "usu_cre","id_person": "id_usucre"})
base1['usu_cre']=base1['usu_cre'].str.strip()
base1["usu_cre"]=base1["usu_cre"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,numdoc,how='left',on='usu_cre')
base1=pd.merge(base1,numdoc,how='inner',on='usu_cre') #Se perdieron unos datos
base1.shape

(68027, 55)

In [39]:
log_falla('id_usucre', 'usu_cre', True)
log_control('dim_personal')
base1=base1.drop('usu_cre',axis=1)

In [40]:
numdoc=numdoc.rename(columns={"usu_cre": "num_doc","id_usucre":"id_numdoc"})
base1['num_doc']=base1['num_doc'].str.strip()
base1["num_doc"]=base1["num_doc"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,numdoc,how='left',on='num_doc')
base1=pd.merge(base1,numdoc,how='left',on='num_doc')
base1.shape

(68027, 55)

In [41]:
log_falla('id_numdoc', 'num_doc', True)
log_control('dim_personal')
base1=base1.drop('num_doc',axis=1)

In [42]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68027 entries, 0 to 68026
Data columns (total 54 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   num_sol       68027 non-null  float64            
 1   fec_sol       68027 non-null  object             
 2   act_med       68027 non-null  float64            
 3   fec_sop       68027 non-null  object             
 4   fec_pro       41810 non-null  object             
 5   hor_pro       37316 non-null  object             
 6   est_fis       45418 non-null  object             
 7   est_sop       68027 non-null  object             
 8   est_reg       68027 non-null  object             
 9   fec_cre       68027 non-null  datetime64[ns, UTC]
 10  usu_mod       41810 non-null  object             
 11  fec_mod       41810 non-null  datetime64[ns, UTC]
 12  est_ocu       41550 non-null  object             
 13  cod_are       68027 non-null  object             
 14  cod_se

In [43]:
numdoc=numdoc.rename(columns={"num_doc": "usu_mod","id_numdoc": "id_usumod"})
base1['usu_mod']=base1['usu_mod'].str.strip()
base1["usu_mod"]=base1["usu_mod"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,numdoc,how='inner',on='usu_mod')
base1=pd.merge(base1,numdoc,how='left',on='usu_mod')
base1.shape

(68027, 55)

In [44]:
log_falla('id_usumod', 'usu_mod', True)
log_control('dim_personal')
base1=base1.drop('usu_mod',axis=1)

In [45]:
numdoc=numdoc.rename(columns={"usu_mod": "eva_num_doc","id_usumod": "id_evanumdoc"})
base1['eva_num_doc']=base1['eva_num_doc'].str.strip()
base1["eva_num_doc"]=base1["eva_num_doc"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,numdoc,how='inner',on='eva_num_doc')
base1=pd.merge(base1,numdoc,how='left',on='eva_num_doc')
base1.shape

(68027, 55)

In [46]:
log_falla('id_evanumdoc', 'eva_num_doc', True)
log_control('dim_personal')
base1=base1.drop('eva_num_doc',axis=1)

In [47]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 68027 entries, 0 to 68026
Data columns (total 54 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   num_sol       68027 non-null  float64            
 1   fec_sol       68027 non-null  object             
 2   act_med       68027 non-null  float64            
 3   fec_sop       68027 non-null  object             
 4   fec_pro       41810 non-null  object             
 5   hor_pro       37316 non-null  object             
 6   est_fis       45418 non-null  object             
 7   est_sop       68027 non-null  object             
 8   est_reg       68027 non-null  object             
 9   fec_cre       68027 non-null  datetime64[ns, UTC]
 10  fec_mod       41810 non-null  datetime64[ns, UTC]
 11  est_ocu       41550 non-null  object             
 12  cod_are       68027 non-null  object             
 13  cod_ser       68027 non-null  object             
 14  ord_op

In [48]:
cqxestfis=cqxestfis.rename(columns={"cod_efi": "est_fis"})
base1['est_fis']=base1['est_fis'].str.strip()
base1["est_fis"]=base1["est_fis"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cqxestfis,how='inner',on='est_fis')
base1=pd.merge(base1,cqxestfis,how='left',on='est_fis') #left, pasó de 68K a 45k
base1.shape

(68027, 56)

In [49]:
log_falla('id_estfis', 'est_fis', True)
log_control('dim_cqxestfis')
base1 = base1.drop(["des_efi","est_fis"],axis=1)

In [50]:
cqxestsolope=cqxestsolope.rename(columns={"cod_eso": "est_sop","id_estsolope": "id_estsop"})
base1['est_sop']=base1['est_sop'].str.strip()
base1["est_sop"]=base1["est_sop"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cqxestsolope,how='left',on='est_sop')
base1=pd.merge(base1,cqxestsolope,how='inner',on='est_sop')
base1.shape

(68027, 56)

In [51]:
log_falla('id_estsop', 'est_sop', True)
log_control('dim_cqxestsolope')
base1 = base1.drop(["des_eso","est_sop"],axis=1)

In [52]:
cqxestreg=cqxestreg.rename(columns={"cod_est": "est_reg"})
base1['est_reg']=base1['est_reg'].str.strip()
base1["est_reg"]=base1["est_reg"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cqxestreg,how='left',on='est_reg')
base1=pd.merge(base1,cqxestreg,how='inner',on='est_reg')
base1.shape

(68027, 56)

In [53]:
log_falla('id_estreg', 'est_reg', True)
log_control('dim_cqxestreg')
base1 = base1.drop(["des_est","est_reg"],axis=1)

In [54]:
base1['cod_are']=base1['cod_are'].str.strip()
base1["cod_are"]=base1["cod_are"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,areas,how='left',on='cod_are')
base1=pd.merge(base1,areas,how='inner',on='cod_are')
base1.shape

(68027, 55)

In [55]:
log_falla('id_area', 'cod_are', True)
log_control('dim_areas')
base1 = base1.drop("cod_are",axis=1)

In [56]:
base1['are_sol']=base1['are_sol'].str.strip()
base1["are_sol"]=base1["are_sol"].str.replace(' ', '', regex=True)
areas=areas.rename(columns={"cod_are": "are_sol","id_area":"id_aresol"})
merge=pd.merge(base1,areas,how='left',on='are_sol')
base1=pd.merge(base1,areas,how='inner',on='are_sol')
base1.shape

(68027, 55)

In [57]:
log_falla('id_aresol', 'are_sol', True)
log_control('dim_areas')
base1 = base1.drop("are_sol",axis=1)

In [58]:
servicios = pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
servicios['cod_ser']=servicios['cod_ser'].str.strip()
base1['cod_ser']=base1['cod_ser'].str.strip()
base1["cod_ser"]=base1["cod_ser"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,servicios,how='left',on='cod_ser')
base1=pd.merge(base1,servicios,how='inner',on='cod_ser')
base1.shape

(68027, 55)

In [59]:
log_falla('id_serv','cod_ser', True)
log_control('dim_servicios')
base1 = base1.drop("cod_ser",axis=1)

In [60]:
base1['ser_sol']=base1['ser_sol'].str.strip()
base1["ser_sol"]=base1["ser_sol"].str.replace(' ', '', regex=True)
servicios=servicios.rename(columns={"id_serv": "id_sersol","cod_ser":"ser_sol"})
merge=pd.merge(base1,servicios,how='left',on='ser_sol')
base1=pd.merge(base1,servicios,how='inner',on='ser_sol')
base1.shape

(68027, 55)

In [61]:
log_falla('id_sersol', 'ser_sol', True)
log_control('dim_servicios')
base1 = base1.drop("ser_sol",axis=1)

In [62]:
base1['eme_sol'].info()

<class 'pandas.core.series.Series'>
Int64Index: 68027 entries, 0 to 68026
Series name: eme_sol
Non-Null Count  Dtype 
--------------  ----- 
0 non-null      object
dtypes: object(1)
memory usage: 1.0+ MB


In [63]:
base1['eme_sol']=base1['eme_sol'].str.strip()
base1["eme_sol"]=base1["eme_sol"].str.replace(' ', '', regex=True)
emecod=emecod.rename(columns={"id_emecod": "id_emesol","cod_eme":"eme_sol"})
merge=pd.merge(base1,emecod,how='inner',on='eme_sol')
base1=pd.merge(base1,emecod,how='left',on='eme_sol') #Aquí usamos left porque no hay datos en eme_sol 
base1.shape

(68027, 56)

In [64]:
log_falla('id_emesol', 'eme_sol', True)
log_control('dim_emecod')
base1 = base1.drop(["des_eme","eme_sol"],axis=1)

In [65]:
base1['ser_eme_sol']=base1['ser_eme_sol'].str.strip()
base1["ser_eme_sol"]=base1["ser_eme_sol"].str.replace(' ', '', regex=True)
emecod=emecod.rename(columns={"id_emesol": "id_seremesol","eme_sol":"ser_eme_sol"})
merge=pd.merge(base1,emecod,how='inner',on='ser_eme_sol')
base1=pd.merge(base1,emecod,how='left',on='ser_eme_sol')#Aquí usamos left porque no hay datos en ser_eme_sol 
base1.shape

(68027, 56)

In [66]:
log_falla('id_seremesol', 'ser_eme_sol', True)
log_control('dim_emecod')
base1 = base1.drop(["ser_eme_sol","des_eme"],axis=1)

In [67]:
base1['tip_ope']=base1['tip_ope'].str.strip()
base1["tip_ope"]=base1["tip_ope"].str.replace(' ', '', regex=True)
cqxtipope=cqxtipope.rename(columns={"cod_ope":"tip_ope"})
merge=pd.merge(base1,cqxtipope,how='inner',on='tip_ope')
base1=pd.merge(base1,cqxtipope,how='left',on='tip_ope') #Se cambió a left porque baja de 68k a 59
base1.shape

(68027, 56)

In [68]:
log_falla('id_tipope', 'tip_ope', True)
log_control('dim_cqxtipope')
base1 = base1.drop(["des_ope","tip_ope"],axis=1)

In [69]:
base1['pri_ope']=base1['pri_ope'].str.strip()
base1["pri_ope"]=base1["pri_ope"].str.replace(' ', '', regex=True)
cqxpriope=cqxpriope.rename(columns={"cod_pop":"pri_ope"})
merge=pd.merge(base1,cqxpriope,how='inner',on='pri_ope')
base1=pd.merge(base1,cqxpriope,how='left',on='pri_ope') #Se cambió a left, porque bajan los datos
base1.shape

(68027, 55)

In [70]:
log_falla('id_priope', 'pri_ope', True)
log_control('dim_cqxpriope')
base1 = base1.drop("pri_ope",axis=1)

In [71]:
base1['rie_qx']=base1['rie_qx'].str.strip()
base1["rie_qx"]=base1["rie_qx"].str.replace(' ', '', regex=True)
cqxrieqx=cqxrieqx.rename(columns={"cod_rqx":"rie_qx"})
merge=pd.merge(base1,cqxrieqx,how='inner',on='rie_qx')
base1=pd.merge(base1,cqxrieqx,how='left',on='rie_qx') #Se cambió a left, porque bajan los datos
base1.shape

(68027, 56)

In [72]:
merge.shape

(41550, 56)

In [73]:
log_falla('id_rieqx', 'rie_qx', True)
log_control('dim_cqxrieqx')
base1 = base1.drop(["rie_qx","des_rqx"],axis=1)

In [74]:
base1['mot_sus']=base1['mot_sus'].str.strip()
base1["mot_sus"]=base1["mot_sus"].str.replace(' ', '', regex=True)
cqxmotsus=cqxmotsus.rename(columns={"cod_msu":"mot_sus"})
merge=pd.merge(base1,cqxmotsus,how='inner',on='mot_sus')
base1=pd.merge(base1,cqxmotsus,how='left',on='mot_sus') #Cambiar a left, se bajan todos los datos
base1.shape

(68027, 56)

In [75]:
log_falla('id_motsus', 'mot_sus', True)
log_control('dim_cqxmotsus')
base1 = base1.drop(["des_msu","mot_sus"],axis=1)

In [76]:
base1['tip_ane']=base1['tip_ane'].str.strip()
base1["tip_ane"]=base1["tip_ane"].str.replace(' ', '', regex=True)
cqxaneste=cqxaneste.rename(columns={"id_anestesia":"id_tipane","cod_ane":"tip_ane"})
merge=pd.merge(base1,cqxaneste,how='inner',on='tip_ane')
base1=pd.merge(base1,cqxaneste,how='left',on='tip_ane') #Tiene que ser left porque no hay datos
base1.shape

(68027, 56)

In [77]:
log_falla('id_tipane', 'tip_ane', True)
log_control('dim_cqxaneste')
base1 = base1.drop(["des_ane","tip_ane"],axis=1)

In [78]:
base1['res_lab']=base1['res_lab'].str.strip()
base1["res_lab"]=base1["res_lab"].str.replace(' ', '', regex=True)
cqxmopreslab=cqxmopreslab.rename(columns={"cod_res":"res_lab"})
merge=pd.merge(base1,cqxmopreslab,how='inner',on='res_lab')
base1=pd.merge(base1,cqxmopreslab,how='left',on='res_lab') #Tiene que ser left porque no hay datos
base1.shape

(68027, 56)

In [79]:
merge.shape

(0, 56)

In [80]:
log_falla('id_reslab', 'res_lab', True)
log_control('dim_cqxmopreslab')
base1 = base1.drop(["des_lab","res_lab"],axis=1)

In [81]:
base1['res_rie_qx']=base1['res_rie_qx'].str.strip()
base1["res_rie_qx"]=base1["res_rie_qx"].str.replace(' ', '', regex=True)
cqxmoprieqx=cqxmoprieqx.rename(columns={"id_rieqx":"id_resrieqx","cod_rie":"res_rie_qx"})
merge=pd.merge(base1,cqxmoprieqx,how='inner',on='res_rie_qx')
base1=pd.merge(base1,cqxmoprieqx,how='left',on='res_rie_qx') #left porque no hay datos
base1.shape

(68027, 56)

In [82]:
log_falla('id_resrieqx', 'res_rie_qx', True)
log_control('dim_cqxmoprieqx')
base1 = base1.drop(["des_rie","res_rie_qx"],axis=1)

In [83]:
base1['res_rie_neu']=base1['res_rie_neu'].str.strip()
base1["res_rie_neu"]=base1["res_rie_neu"].str.replace(' ', '', regex=True)
cqxmoprieneu=cqxmoprieneu.rename(columns={"id_rieneu":"id_resrieneu","cod_rieneu":"res_rie_neu"})
merge=pd.merge(base1,cqxmoprieneu,how='inner',on='res_rie_neu')
base1=pd.merge(base1,cqxmoprieneu,how='left',on='res_rie_neu') #Left porque no hay datos
base1.shape

(68027, 56)

In [84]:
log_falla('id_resrieneu', 'res_rie_neu', True)
log_control('dim_cqxmoprieneu')
base1 = base1.drop(["des_rieneu","res_rie_neu"],axis=1)

In [85]:
base1['con_inf']=base1['con_inf'].str.strip()
base1["con_inf"]=base1["con_inf"].str.replace(' ', '', regex=True)
cqxmopconinf=cqxmopconinf.rename(columns={"cod_coninf":"con_inf"})
merge=pd.merge(base1,cqxmopconinf,how='inner',on='con_inf')
base1=pd.merge(base1,cqxmopconinf,how='left',on='con_inf') #Left porque no hay datos
base1.shape

(68027, 56)

In [86]:
log_falla('con_inf', 'con_inf', True)
log_control('dim_cqxmopconinf')
base1 = base1.drop(["des_coninf","con_inf"],axis=1)

In [87]:
base1['ord_tra']=base1['ord_tra'].str.strip()
base1["ord_tra"]=base1["ord_tra"].str.replace(' ', '', regex=True)
cqxmopordtra=cqxmopordtra.rename(columns={"cod_ordtra":"ord_tra"})
merge=pd.merge(base1,cqxmopordtra,how='inner',on='ord_tra') 
base1=pd.merge(base1,cqxmopordtra,how='left',on='ord_tra') #Left porque no hay datos
base1.shape

(68027, 56)

In [88]:
log_falla('id_ordtra', 'ord_tra', True)
log_control('dim_cqxmopordtra')
base1 = base1.drop(["des_ordtra","ord_tra"],axis=1)

In [89]:
base1['res_eva']=base1['res_eva'].str.strip()
base1["res_eva"]=base1["res_eva"].str.replace(' ', '', regex=True)
cqxresevapreqx=cqxresevapreqx.rename(columns={"cod_reseva":"res_eva"})
merge=pd.merge(base1,cqxresevapreqx,how='inner',on='res_eva')
base1=pd.merge(base1,cqxresevapreqx,how='left',on='res_eva') #Left porque no hay muchos datos
base1.shape

(68027, 56)

In [90]:
merge.shape

(7493, 56)

In [91]:
log_falla('id_reseva', 'res_eva', True)
base1 = base1.drop(["des_reseva","res_eva"],axis=1)
dim.append('dim_cqxresevapreqx')
control_d.append(base1.shape[0])

In [92]:
base1['fec_sol'] = pd.to_datetime(base1['fec_sol']).dt.date
tiempo=tiempo.rename(columns={"fec_pro":"fec_sol"})
merge = pd.merge(base1, tiempo, how='inner', on='fec_sol')
base1 = pd.merge(base1, tiempo, how='left', on='fec_sol') #Puede ser inner
base1=base1.rename(columns={"id_time_pro":"id_time_sol","ano_pro":"ano_sol","mes_pro":"mes_sol",
                            "dia_pro":"dia_sol","dia_sem_pro":"dia_sem_sol","sem_pro":"sem_sol"})

In [93]:
base1.shape

(68027, 60)

In [94]:
base1['fec_sop_temp'] = base1['fec_sop'].astype(str).str.split().str[0]
tiempo=tiempo.rename(columns={"fec_sol":"fec_sop_temp"})
tiempo["fec_sop_temp"] = tiempo['fec_sop_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_sop_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_sop_temp')
base1=base1.rename(columns={"id_time_pro":"id_time_sop","ano_pro":"ano_sop","mes_pro":"mes_sop",
                            "dia_pro":"dia_sop","dia_sem_pro":"dia_sem_sop","sem_pro":"sem_sop"})
base1=base1.drop("fec_sop_temp",axis=1)
base1.shape

(68027, 66)

In [95]:
base1['fec_cre_temp'] = base1['fec_cre'].astype(str).str.split().str[0]
tiempo=tiempo.rename(columns={"fec_sop_temp":"fec_cre_temp"})
tiempo["fec_cre_temp"] = tiempo['fec_cre_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_cre_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_cre_temp')
base1=base1.rename(columns={"id_time_pro":"id_time_cre","ano_pro":"ano_cre","mes_pro":"mes_cre",
                            "dia_pro":"dia_cre","dia_sem_pro":"dia_sem_cre","sem_pro":"sem_cre"})
base1=base1.drop("fec_cre_temp",axis=1)
base1.shape

(68027, 72)

In [96]:
base1['fec_mod_temp'] = base1['fec_mod'].astype(str).str.split().str[0]
tiempo = tiempo.rename(columns={"fec_cre_temp": "fec_mod_temp"})
tiempo["fec_mod_temp"] = tiempo['fec_mod_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_mod_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_mod_temp')
base1=base1.rename(columns={"ano_pro":"ano_mod","mes_pro":"mes_mod",
                            "dia_pro":"dia_mod","dia_sem_pro":"dia_sem_mod","sem_pro":"sem_mod"})
base1=base1.drop(["fec_mod_temp","id_time_pro"], axis=1)
base1.shape

(68027, 77)

In [97]:
base1['fec_eva_temp'] = base1['fec_eva'].astype(str)
tiempo = tiempo.rename(columns={"fec_mod_temp": "fec_eva_temp"})
tiempo["fec_eva_temp"] = tiempo['fec_eva_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_eva_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_eva_temp')
base1=base1.rename(columns={"id_time_pro":"id_time_eva","ano_pro":"ano_eva","mes_pro":"mes_eva",
                            "dia_pro":"dia_eva","dia_sem_pro":"dia_sem_eva","sem_pro":"sem_eva"})
base1=base1.drop("fec_eva_temp", axis=1)
base1.shape

(68027, 83)

In [98]:
base1['fec_sol_temp'] = base1['fec_sol_exa'].astype(str)
tiempo = tiempo.rename(columns={"fec_eva_temp": "fec_sol_temp"})
tiempo["fec_sol_temp"] = tiempo['fec_sol_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_sol_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_sol_temp')
base1 = base1.rename(columns={"id_time_pro": "id_time_solexa", "ano_pro": "ano_solexa", "mes_pro": "mes_solexa", 
                              "dia_pro": "dia_solexa", "dia_sem_pro": "dia_sem_solexa", "sem_pro": "sem_solexa"})
base1=base1.drop("fec_sol_temp", axis=1)
base1.shape

(68027, 89)

In [99]:
base1['fec_res_exa_temp'] = base1['fec_res_exa'].astype(str)
tiempo = tiempo.rename(columns={"fec_sol_temp": "fec_res_exa_temp"})
tiempo["fec_res_exa_temp"] = tiempo['fec_res_exa_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_res_exa_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_res_exa_temp')
base1 = base1.rename(columns={"id_time_pro": "id_timeresexa", "ano_pro": "ano_resexa", "mes_pro": "mes_resexa", 
                              "dia_pro": "dia_resexa", "dia_sem_pro": "dia_sem_resexa", "sem_pro": "sem_resexa"})
base1=base1.drop("fec_res_exa_temp", axis=1)
base1.shape

(68027, 95)

In [100]:
base1['fec_rie_car_temp'] = base1['fec_rie_car'].astype(str)
tiempo = tiempo.rename(columns={"fec_res_exa_temp": "fec_rie_car_temp"})
tiempo["fec_rie_car_temp"] = tiempo['fec_rie_car_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_rie_car_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_rie_car_temp')
base1 = base1.rename(columns={"id_time_pro": "id_time_riecar", "ano_pro": "ano_riecar", "mes_pro": "mes_riecar", 
                              "dia_pro": "dia_riecar", "dia_sem_pro": "dia_sem_riecar", "sem_pro": "sem_riecar"})
base1=base1.drop("fec_rie_car_temp", axis=1)
base1.shape

(68027, 101)

In [101]:
base1['fec_rie_neu_temp'] = base1['fec_rie_neu'].astype(str)
tiempo = tiempo.rename(columns={"fec_rie_car_temp": "fec_rie_neu_temp"})
tiempo["fec_rie_neu_temp"] = tiempo['fec_rie_neu_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_rie_neu_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_rie_neu_temp')
base1 = base1.rename(columns={"id_time_pro": "id_time_rieneu", "ano_pro": "ano_rieneu", "mes_pro": "mes_rieneu",
                              "dia_pro": "dia_rieneu", "dia_sem_pro": "dia_sem_rieneu", "sem_pro": "sem_rieneu"})
base1=base1.drop("fec_rie_neu_temp", axis=1)
base1.shape

(68027, 107)

In [102]:
base1['fec_pro_temp'] = base1['fec_pro'].astype(str)
tiempo = tiempo.rename(columns={"fec_rie_neu_temp": "fec_pro_temp"})
tiempo["fec_pro_temp"] = tiempo['fec_pro_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='fec_pro_temp')
base1 = pd.merge(base1, tiempo, how='left', on='fec_pro_temp')
base1=base1.drop("fec_pro_temp", axis=1)
base1.shape

(68027, 113)

In [103]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [104]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [105]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [106]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [107]:
base.to_sql(name=f'{dat}', con=connection2, if_exists='append', index=False,chunksize=10000)

6027

In [108]:
fecha_fin = "2024-12-31"

In [110]:
#proceso = "DAT"
#cod_proceso = 13

proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=13", con=connection2)
proceso = proceso.iloc[0, 0]
cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=13", con=connection2)
cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado
nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']
tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [111]:
tabla_logs

,codigo,proceso,dat,fecha_ejecucion,hora_inicio,hora_fin,dim,fecha_ini,fecha_ter,esperado,obtenido,faltante,falla,id_afectado
0,13,DAT,dat_cqxt001_essi,2023-06-16,16:05,16:07,dim_cqxcenqx,2023-05-01,2024-12-31,68485,68484,1,1005None,id_cqx
1,13,DAT,dat_cqxt001_essi,2023-06-16,16:05,16:07,dim_cqxsalas,2023-05-01,2024-12-31,68484,68484,0,NaN,id_sala
2,13,DAT,dat_cqxt001_essi,2023-06-16,16:05,16:07,dim_oricas,2023-05-01,2024-12-31,68484,68484,0,0,id_oricas
3,13,DAT,dat_cqxt001_essi,2023-06-16,16:05,16:07,dim_oricas,2023-05-01,2024-12-31,68484,68484,0,0,id_oricaseva
4,13,DAT,dat_cqxt001_essi,2023-06-16,16:05,16:07,dim_cas,2023-05-01,2024-12-31,68484,68484,0,0,id_cas
5,13,DAT,dat_cqxt001_essi,2023-06-16,16:05,16:07,dim_cas,2023-05-01,2024-12-31,68484,68484,0,0,id_caseva
6,13,DAT,dat_cqxt001_essi,2023-06-16,16:05,16:07,dim_cas,2023-05-01,2024-12-31,68484,68484,0,0,id_red
7,13,DAT,dat_cqxt001_essi,2023-06-16,16:05,16:07,dim_tipdoc,2023-05-01,2024-12-31,68484,68484,0,0,id_tdi
8,13,DAT,dat_cqxt001_essi,2023-06-16,16:05,16:07,dim_tipdoc,2023-05-01,2024-12-31,68484,68484,0,None,id_evatipdoc
9,13,DAT,dat_cqxt001_essi,2023-06-16,16:05,16:07,dim_personal,2023-05-01,2024-12-31,68484,68027,457,"JPAREDES, MEDICOREG, 41074955N, 421449081, ...",id_usucre


In [ ]:
tabla_logs.to_sql(name=f'logs', con=connection4, if_exists='append', index=False,chunksize=10000)

In [ ]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3) #Redondea el tiempo de proceso a 3 decimales.
print("Proceso 1.3 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

In [ ]:
connection1.close()
connection2.close()
connection3.close()

In [ ]:
engine1.dispose()
engine2.dispose()
engine3.dispose()